# Binary GP Classification と Active Learning

2値分類で、latent GP と class probability を分離して確認し、分類専用 acquisition から次にラベル付けする候補を生成します。

## このNotebookで確認すること

- Bernoulli likelihood を持つ variational GP classifier の構築と学習
- latent posterior と class probability の違い
- latent / probability / label uncertainty の違い
- Predictive Entropy と BALD による Active Learning
- BoTorch `optimize_acqf` による q=1 candidate generation

関連理論: [Gaussian Process Classification](../../docs/theory/24_classification.md) / [Classification Active Learning](../../docs/theory/acquisition/13_classification_active_learning.md)

```text
Data -> Model -> Variational fit -> Latent posterior -> Class probability
     -> Classification acquisition -> optimize_acqf -> Candidate -> Label evaluation
```

## 1. Imports and reproducibility

### ここでのポイント

`torch.manual_seed(0)` は乱数を固定して結果の再現性を高めます。ただし、計算環境・依存ライブラリによって完全に同じ結果になる保証はありません。

`torch.double` は倍精度浮動小数点（float64）です。GPの数値計算では有用ですが、既存テンソルのdtypeを自動変換するわけではありません。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.optim import optimize_acqf

from robotorchan.acquisition import BALD, PredictiveEntropy, ProbabilityVariance
from robotorchan.models.classification import BinarySingleTaskGPClassifier

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Binary classification problem

潜在境界が分かる1次元の合成問題を使います。観測ラベルは0/1です。

### データを読む

- `train_X` は6件×1特徴量の入力行列で、形状は `(6, 1)` です。
- `train_Y` は0/1ラベルが6個並ぶベクトルです。境界 `x=0.5` より右側がクラス1という**人工的な正解ルール**を使用します。
- `bounds` は特徴量の下限と上限を指定します。ここでは0〜1です。

実データでは `true_label` のような正解関数を通常は知りません。候補を測定・実験して初めてラベルを得ます。

In [ ]:
def true_label(X: torch.Tensor) -> torch.Tensor:
    return (X.squeeze(-1) > 0.5).to(dtype=torch.double)


train_X = torch.tensor([[0.05], [0.15], [0.30], [0.70], [0.85], [0.95]])
train_Y = true_label(train_X)
bounds = torch.tensor([[0.0], [1.0]])

print(train_X.shape, train_Y.shape)
print(train_Y)

## 3. Model construction

`posterior(X)` はclass probabilityではなくlatent function posteriorです。

### モデルの各引数

`train_X` / `train_Y` は観測済みデータです。`inducing_points=6` は変分GPで使う誘導点数です。大きくすれば必ず精度が向上するとは限らず、計算負荷にも影響します。

`classification_metadata` はモデルの分類に関する設定確認、`make_mll()` は学習目標の種類を確認するために表示しています。

In [ ]:
model = BinarySingleTaskGPClassifier(
    train_X,
    train_Y,
    inducing_points=6,
)

print(model.classification_metadata)
print(model.raw_train_X.shape, model.raw_train_Y.shape)
print(type(model.make_mll()).__name__)

## 4. Variational fitting

Bernoulli likelihoodなのでExactMarginalLogLikelihoodではなくVariationalELBOを学習します。`fit()`は内部でBernoulli ELBOを最適化します。


### 変分学習とは？

回帰の典型的なGPではガウス尤度を仮定できます。一方、0/1ラベルを使うBernoulli尤度では、一般に厳密な周辺尤度の計算が困難です。ここでは **ELBO（Evidence Lower Bound）** を最適化する近似推論を使います。

`epochs=40` はデモの実行時間を抑える設定で、最適な学習回数ではありません。`lr=0.05` は学習率です。損失の収束、ホールドアウト評価、乱数シードを変えた頑健性などを確認して調整します。

In [ ]:
model.fit(epochs=40, lr=0.05)
print("Binary GP classifier fitted")


## 5. Latent posterior と class probability

latent meanは確率ではありません。`predict_proba`はlikelihoodを通してlatent uncertaintyを積分した予測確率です。

### 潜在関数とクラス確率の見分け方

- `latent_mean` は実数値であり、0〜1の範囲に制限されません。
- `latent_std` は潜在関数の標準偏差で、予測がどの程度広がるかを表します。
- `probability` は **クラス1の予測確率** です。

`grid` は0〜1の201点で、形状は `(201, 1)`。形状を出力しているのは、次元の取り違えを防ぐためです。

In [ ]:
grid = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
with torch.no_grad():
    latent = model.latent_posterior(grid)
    latent_mean = latent.mean.squeeze(-1)
    latent_std = latent.variance.clamp_min(0.0).sqrt().squeeze(-1)
    probability_posterior = model.probability_posterior(grid)
    probability = probability_posterior.mean[..., 1]
    lower, upper = probability_posterior.credible_interval(0.95)
    probability_lower = lower[..., 1]
    probability_upper = upper[..., 1]
    torch.testing.assert_close(probability, model.predict_proba(grid)[..., 1])

print("latent:", latent_mean.shape)
print("probability mean / 95% credible limits:", probability.shape, lower.shape)


## 6. 3種類の不確実性

`latent_variance` は潜在関数の分散、`probability_posterior.variance` は**クラス確率そのものの事後分散**、
`predictive_variance` は**観測する0/1ラベルのBernoulli分散**です。
混同しないよう、以下では同じ入力点で比較します。

### 潜在GP、確率Posterior、観測ラベルの違い

1. **潜在分散**：ガウス事後分布 `Var(f(x)|D)`。その平均は0〜1の確率ではありません。
2. **クラス確率の分散**：`Var(P(y=1|f(x))|D)`。二値ProbitではGauss–Hermite求積で評価します。
3. **観測ラベルの分散**：`P(y=1|x,D) × (1-P(y=1|x,D))`。確率Posteriorの分散とは別の値です。
4. **95%信用区間**：潜在ガウス分位点をProbitリンクで変換した**確率そのものの周辺信用区間**。
   この例ではサンプリング不要です。観測ラベル0/1の「予測区間」ではありません。

`posterior(X)` は潜在GP、`probability_posterior(X)` はクラス確率のPosteriorです。
`mean`、`variance`、`rsample`、`credible_interval` を分けて使います。

In [ ]:
with torch.no_grad():
    latent_variance = model.latent_variance(grid).squeeze(-1)
    # Gauss–Hermite integration: epistemic uncertainty about P(y=1|f).
    probability_variance = probability_posterior.variance[..., 1]
    # Bernoulli observation variance: intrinsic label uncertainty is distinct.
    label_variance = model.predictive_variance(grid)[..., 1]
    label_entropy = model.predictive_entropy(grid)

torch.testing.assert_close(label_variance, probability * (1.0 - probability))
print("latent variance:", latent_variance.shape)
print("probability variance:", probability_variance.shape)
print("label variance / entropy:", label_variance.shape, label_entropy.shape)


## 7. Classification Active Learning

Predictive Entropyはlabel ambiguity、BALDはそのうちlatent/model uncertaintyに由来する情報量を評価します。

### 獲得関数の入力形状

`grid` は `(201, 1)` ですが、`Xq = grid.unsqueeze(-2)` によって `(201, 1, 1)` にします。順番に **候補数、q（同時候補点数）、特徴量数** です。

EntropyとBALDのスコアはそれぞれ独立に計算されます。値が高い場所ほど優先されますが、**両獲得関数のスコアの大小を直接比較するものではありません**。

In [ ]:
entropy_acq = PredictiveEntropy(model)
variance_acq = ProbabilityVariance(model)
bald_acq = BALD(model)  # Binary Probit uses quadrature (not MC by default).

Xq = grid.unsqueeze(-2)  # [201, q=1, d=1]
with torch.no_grad():
    entropy_score = entropy_acq(Xq)
    variance_score = variance_acq(Xq)
    bald_score = bald_acq(Xq)
    torch.testing.assert_close(
        variance_score, model.probability_variance(Xq).mean(dim=-1)
    )
    torch.testing.assert_close(bald_score, model.mutual_information(Xq))

print("Entropy / ProbabilityVariance / BALD:", entropy_score.shape,
      variance_score.shape, bald_score.shape)


## 8. Candidate generation

**このNotebookでは次の測定点を1点だけ提案するため、`q=1` を使用します。**
`PredictiveEntropy`・`MarginUncertainty`・`ProbabilityVariance`・`BALD` は単一点用の獲得関数です。
一方、robotorchanには `qPredictiveEntropy`・`qMarginUncertainty`・`qProbabilityVariance`・`qBALD` などのバッチ版もあります。
前者3つのq版は各候補の**周辺スコアの平均**を使い、`qBALD` は候補間の依存を含む同時ラベル情報利得をMC推定します。
ここでは単一点用 `PredictiveEntropy` を標準BoTorchの `optimize_acqf` に渡します。

q-batchの説明は[分類Active Learningガイド](../../docs/guide/acquisition/13_classification_active_learning.md)を参照してください。

### 候補を選ぶ処理

`optimize_acqf` は獲得関数を最大化して次の1点を返します。`num_restarts=5` は最適化の再開始回数、`raw_samples=64` は初期候補の探索に使うサンプル数です。

`next_label = true_label(...)` は合成問題だから可能な**疑似測定**です。現実の実験ではこの行の代わりに実験設備や測定者から結果を受け取ります。

In [ ]:
candidate, acquisition_value = optimize_acqf(
    acq_function=entropy_acq,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
next_label = true_label(candidate.detach()).to(dtype=torch.long)

print("candidate:", candidate.detach().squeeze().item())
print("acquisition value:", acquisition_value.detach().item())
print("observed label:", next_label.item())

## 9. Visualization

### まず2つのグラフを読む

上段はクラス1確率と実測ラベル、下段は獲得関数スコアです。破線はEntropyによって選ばれた候補です。実測ラベルの点が0または1でも、予測確率はその間の連続値になります。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].plot(grid.squeeze(-1), probability, label="E[P(y=1|f)]")
axes[0].fill_between(
    grid.squeeze(-1), probability_lower, probability_upper,
    alpha=0.2, label="95% credible interval of P(y=1|f)",
)
axes[0].scatter(train_X.squeeze(-1), train_Y, label="observations")
axes[0].axvline(candidate.detach().squeeze().item(), linestyle="--", label="next candidate")
axes[0].set_ylabel("probability / label")
axes[0].legend()

axes[1].plot(grid.squeeze(-1), entropy_score, label="PredictiveEntropy")
axes[1].plot(grid.squeeze(-1), bald_score, label="BALD")
axes[1].set_xlabel("x")
axes[1].set_ylabel("acquisition")
axes[1].legend()
plt.show()

## 9.1 可視化で理解する：潜在関数・確率・不確実性

下の4つの図は、**同じ入力点**を異なる尺度で見ています。左上の潜在関数の平均と不確実性の帯は、右上の0〜1の予測確率とは異なります。右上の帯は確率Posteriorの95%信用区間です。左下は「確率そのものの揺れ」、右下は「次のラベルの曖昧さと情報獲得」を示します。

各曲線の最大値が一致する必要はありません。とくにBALDとEntropyが選ぶ点の違いを観察してください。

In [ ]:
x_plot = grid.squeeze(-1).detach().cpu().numpy()
latent_mean_plot = latent_mean.detach().cpu().numpy()
latent_std_plot = latent_std.detach().cpu().numpy()
p_plot = probability.detach().cpu().numpy()
p_lower_plot = probability_lower.detach().cpu().numpy()
p_upper_plot = probability_upper.detach().cpu().numpy()
label_var_plot = label_variance.detach().cpu().numpy()
var_plot = probability_variance.detach().cpu().numpy()
entropy_plot = entropy_score.detach().cpu().reshape(-1).numpy()
bald_plot = bald_score.detach().cpu().reshape(-1).numpy()
observed_x = train_X.squeeze(-1).detach().cpu().numpy()
observed_y = train_Y.detach().cpu().numpy()
selected_x = candidate.detach().cpu().reshape(-1)[0].item()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes[0, 0].plot(x_plot, latent_mean_plot, label="latent mean")
axes[0, 0].fill_between(
    x_plot,
    latent_mean_plot - 2 * latent_std_plot,
    latent_mean_plot + 2 * latent_std_plot,
    alpha=0.2,
    label="mean ± 2 std",
)
axes[0, 0].axhline(0, linestyle=":", color="gray")
axes[0, 0].set_title("Latent function: not a probability")
axes[0, 0].legend()

axes[0, 1].plot(x_plot, p_plot, label="mean P(y=1)")
axes[0, 1].fill_between(
    x_plot, p_lower_plot, p_upper_plot, alpha=0.2, label="95% credible interval"
)
axes[0, 1].scatter(observed_x, observed_y, label="observed label")
axes[0, 1].axhline(0.5, linestyle=":", color="gray")
axes[0, 1].set_ylim(-0.05, 1.05)
axes[0, 1].set_title("Predicted class probability")
axes[0, 1].legend()

axes[1, 0].plot(x_plot, var_plot, label="epistemic variance of class probability")
axes[1, 0].plot(x_plot, label_var_plot, linestyle="--", label="Bernoulli label variance")
axes[1, 0].set_title("Uncertainty about class probability")
axes[1, 0].set_xlabel("x")
axes[1, 0].legend()

axes[1, 1].plot(x_plot, entropy_plot, label="Predictive Entropy")
axes[1, 1].plot(x_plot, bald_plot, label="BALD")
axes[1, 1].set_title("Active Learning scores")
axes[1, 1].set_xlabel("x")
axes[1, 1].legend()

for ax in axes.flat:
    ax.axvline(selected_x, linestyle="--", color="gray", alpha=0.7)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 図から読み取ること

- 潜在平均のグラフには0〜1の制約はありません。左上の帯は潜在Gaussianの約95%範囲、右上の帯はクラス確率の95%信用区間です。異なる確率変数を表します。
- クラス確率が0.5付近なら予測ラベルは曖昧ですが、モデルパラメータの不確実性が最大とは限りません。
- 破線はEntropyで実際に最適化した候補位置です。BALDが最大の点と同じとは限りません。
- ここではGP分類が滑らかな境界を学習するため、人工的な0/1ステップ関数を完全に再現しない場合があります。

## 10. 使い所と注意点

- 通常の2値分類では `BinarySingleTaskGPClassifier` を入口にします。
- `posterior()` はlatent posteriorで、class probabilityではありません。
- Probability VarianceとBernoulli predictive varianceは別の量です。95%信用区間はクラス確率の範囲です。
- 単一点のProbabilityVariance獲得値はクラス確率Posteriorのクラス平均分散と一致します。
- 回帰用qEIやStraddleを、Gaussian latent posteriorという理由だけで分類へ流用しません。
- このNotebookは`q=1`の単一点提案例です。ライブラリ全体は`q>1`の分類Active Learningにも対応し、`qBALD`以外の周辺型q獲得関数は各点のスコアを平均します。
- Entropy / Margin / Probability Variance / BALDの契約はmulticlassへ拡張可能ですが、Latent Straddleはbinary固有です。

このNotebookは候補を1点生成して真のlabelを評価するところまでを示します。実運用では、その観測をtraining dataへ追加して再fitする処理を反復します。

## 初学者向け補足：分類と回帰の違い

回帰は連続値（例：温度や強度）を予測し、二値分類は「合格／不合格」「正常／異常」のような **0/1ラベル** を予測します。分類器が返す確率 0.8 は「予測値が0.8」という意味ではなく、モデルの仮定のもとでクラス1になる確率が約80%という意味です。

### GP分類で何が起きているか

1. 入力 `X` から、観測できない連続的な **潜在関数** `f(X)` をGPで推定します。
2. Bernoulli尤度のリンク関数を介して、潜在関数を0〜1のクラス確率へ変換します。
3. 0/1の観測ラベルを用いて学習します。尤度が非ガウスなので、このNotebookでは変分推論（ELBO）を使用します。
4. `posterior(X)` は潜在関数の分布、`predict_proba(X)` はクラス確率の推定です。両者を混同しないでください。

### Active Learningで次の点を選ぶ理由

- **Predictive Entropy**：予測ラベルがどちらになるか曖昧な点を重視します。0.5付近で大きくなります。
- **BALD**：モデルの知識不足に由来する不確実性（epistemic uncertainty）を重視します。ラベル自体の偶然性とは区別します。
- 候補選択は「クラス1の確率が最大の点を選ぶ」こととは異なります。目的に合った獲得関数を選びます。

### 結果を読むときのチェック

学習点の近くで予測がどう変わるか、未観測領域で不確実性がどう増えるか、選択候補が境界付近に集中するかを図で確認してください。境界付近に集中しない場合も、獲得関数や学習データによっては正常です。

**よくある間違い：** ラベルを連続的な回帰ターゲットとして扱う、潜在平均をそのまま確率として読む、`X` の次元と候補バッチ次元を混同する、学習データで測った精度を汎化性能とみなす。